# Reproduce: A1_baseline_BPTT_864p

Notebook self-contained per ri-eseguire la training run originale.

**Architettura**: variant `baseline` (864 params)
**Run originale**: vedi `snapshot_original/training_log.csv` (30 epochs)
**Snapshot ep1**: val_total=0.3945491317104786, val_data=0.3794287653996589

**Setup CLI**: vedi `README.md` per il comando completo.

## Celle
- **Cell 1**: ENV check + `build_model('baseline')` + verify 864 params
- **Cell 2**: smoke 1 ep × 1 step + diff numerico vs snapshot ep1
- **Cell 3** (opzionale): full reproduction 30 ep (richiede cache `data/cache_1500_highway_cut0.0_ou0.0.pt`)


In [ ]:
# Cell 1 -- ENV check + build_model verify
import sys, os

# Verifica file critici presenti
for f in ['core/network.py', 'core/neurons.py', 'core/hardware.py',
          'config.py', 'train.py', 'data/generator.py',
          'snapshot_original/training_log.csv',
          'snapshot_original/config_snapshot.json']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

# Build model + verify params
from core.network import build_model
m = build_model('baseline')
n = sum(p.numel() for p in m.parameters())
print(f'\n  Built: {type(m).__name__}  params={n:,}  (expected 864)')
assert n == 864, f'PARAMS MISMATCH: {n} vs 864'
print('\n[OK] ENV check passed')


In [ ]:
# Cell 2 -- Smoke 1 ep x 1 step + numeric diff vs snapshot
import subprocess, sys, csv, os, math

TAG = 'A1_baseline_BPTT_864p_smoke_1ep'

args = [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', '1',
        '--max_steps_per_epoch', '1',
        '--batch_size', '8',
        '--val_batch_size', '64',
        '--seq_len', '50',
        '--scheduler', 'onecycle',
        '--max_lr', '0.002',
        '--lr', '0.002',
        '--optimizer', 'adamw',
        '--prodigy_d_coef', '1.0',
        '--scenario_mix', 'highway',
        '--cut_in_ratio', '0.0',
        '--cf_hidden_size', '32',
        '--cf_rank', '8',
        '--lambda_data', '1.0',
        '--lambda_phys', '0.1',
        '--lambda_ou', '0.05',
        '--lambda_bc', '1.0',
        '--lambda_sr', '0.0',
        '--noise_scale', '0.0',
        '--po2_enabled', '1',
        '--max_inf_streak', '99999',
        '--early_stop_patience', '0',
        '--tag', TAG]
print('CLI:', ' '.join(args[1:]))

r = subprocess.run(args, capture_output=True, text=True)
if r.returncode != 0:
    print('STDERR:\n', r.stderr[-2000:])
    raise RuntimeError(f'train.py failed rc={r.returncode}')

log_smoke = f'checkpoints/{TAG}/training_log.csv'
assert os.path.isfile(log_smoke), 'log smoke missing'

rows_smoke = list(csv.DictReader(open(log_smoke)))
rows_orig = list(csv.DictReader(open('snapshot_original/training_log.csv')))

smoke_ep1 = rows_smoke[0]
orig_ep1 = rows_orig[0]

print(f'\n--- Smoke ep1 ---')
for k in ['val_total', 'val_data', 'val_phys', 'val_ou', 'val_bc', 'spike_rate']:
    sv = float(smoke_ep1.get(k, 'nan'))
    ov = float(orig_ep1.get(k, 'nan'))
    rel = abs(sv - ov) / max(abs(ov), 1e-6) * 100 if not math.isnan(sv) and not math.isnan(ov) else float('nan')
    flag = '[OK]' if not math.isnan(rel) and rel < 50 else '[DIVERGE]'
    print(f'  {flag} {k:<12} smoke={sv:.4f}  orig={ov:.4f}  diff_rel={rel:.1f}%')

print('\nNote: smoke usa max_steps_per_epoch=1, quindi diff puo essere significativa.')
print('Cosa contare: i numeri sono FINITI e nello stesso ordine di grandezza.')


In [ ]:
# Cell 3 (OPZIONALE) -- Full reproduction
# NB: richiede cache disponibile + tempo significativo (vedi README per stima)
RUN_FULL = False  # Cambia a True per eseguire

if RUN_FULL:
    import subprocess, sys
    TAG = 'A1_baseline_BPTT_864p_full_reproduce'
    CACHE_PATH = 'data/cache_1500_highway_cut0.0_ou0.0.pt'
    args = [sys.executable, 'train.py',
            '--training_method', 'baseline',
        '--epochs', '30',
        '--max_steps_per_epoch', '190',
        '--batch_size', '8',
        '--val_batch_size', '64',
        '--seq_len', '50',
        '--scheduler', 'onecycle',
        '--max_lr', '0.002',
        '--lr', '0.002',
        '--optimizer', 'adamw',
        '--prodigy_d_coef', '1.0',
        '--scenario_mix', 'highway',
        '--cut_in_ratio', '0.0',
        '--cf_hidden_size', '32',
        '--cf_rank', '8',
        '--lambda_data', '1.0',
        '--lambda_phys', '0.1',
        '--lambda_ou', '0.05',
        '--lambda_bc', '1.0',
        '--lambda_sr', '0.0',
        '--noise_scale', '0.0',
        '--po2_enabled', '1',
        '--max_inf_streak', '99999',
        '--early_stop_patience', '0',
            '--data_cache', CACHE_PATH,
            '--tag', TAG]
    print('Running full reproduction. Stima tempo: vedi README.')
    r = subprocess.run(args)
    print(f'Exit code: {r.returncode}')
    print(f'Results in: checkpoints/{TAG}/')
else:
    print('RUN_FULL=False — skip. Imposta a True per eseguire la full run.')
